# Synthetic Data Augmentation with Conditional Flow Matching

## Part 3

In this section, we evaluate the impact of incorporating synthetic image data on classification performance. In particular, we examine changes in classification accuracy and score when the Fashion MNIST dataset is augmented with synthetic samples in the extremely low-data regime, using 0.1% of the available data for model training.

## Setup

In [1]:
!find . -mindepth 1 -exec rm -rf {} + &> /dev/null
!git clone https://github.com/ZhangLyndon/FlowMatchingAugmentation . > /dev/null 2>&1

In [2]:
!pip install -qU pip
!pip install -qU -r requirements.txt

In [3]:
import os
import sys
import argparse
import functools

# Silence tqdm output
os.environ["TQDM_DISABLE"] = "1"

# Reduce CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
import torch
import torchvision
import numpy as np

# Components for initializing an ImageNet-pretrained ResNet-18 classifier, fine-
# tuning it on Fashion MNIST, and evaluating classification performance on base-
# line, low-data, and synthetically augmented settings.
from classification import (ClassificationTrainer,
                            create_classifier, ResNetClassifier,
                            SyntheticDataGenerator, SyntheticAugmentationEvaluator,
                            create_augmented_dataset)

# Utilities for loading the Fashion MNIST dataset, computing top-k categorical
# accuracy and average cross-entropy loss, and saving training results.
from utils import get_dataloaders, AverageMeter, accuracy, save_results

import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["font.family"] = "DejaVu Sans Mono"

We assess the performance impact of synthetic augmentation in the extremely low-data regime (using 0.1% of the training set), and begin by evaluating the classification performance of the unaugmented training split, measuring both accuracy and the macro $\mathsf F_1$ score.

In [5]:
# Configure augmentation evaluation pipeline
augmentation_args = argparse.Namespace(data_root = "./data",
                                       batch_size = 16,
                                       num_workers = 0,
                                       epochs = 25,
                                       lr = 0.001,
                                       weight_decay = 1e-4,
                                       step_size = 15,
                                       gamma = 0.1,
                                       synthetic_data_dir = "./images",
                                       real_ratio = 0.001,
                                       classification_dir = "./results/classification",
                                       augmentation_dir = "./results/augmentation",
                                       checkpoint_dir = "./checkpoints",
                                       save_interval = 20,
                                       seed = 42)

# Create directory to store synthetic augmentation results
os.makedirs(augmentation_args.augmentation_dir, exist_ok = True)

# Set random seed for reproducibility
torch.manual_seed(augmentation_args.seed)
np.random.seed(augmentation_args.seed)

In [6]:
guidance_scale = 3.0
evaluator = SyntheticAugmentationEvaluator(augmentation_args, guidance_scale)
evaluator.run_low_data_experiments(augmentation_args.real_ratio, False)

Number of epochs: 25
Number of training samples: 60
Number of validation samples: 10000
Epoch 1/25
Training Set | Loss: 2.7557, Top-1 Accuracy: 10.00%, Top-5 Accuracy: 55.00%
Validation Set | Loss: 2.4488, Top-1 Accuracy: 15.94%, Top-5 Accuracy: 58.38%
Best Validation Loss (Up Until Now): 2.4488
_________________________________________________________________________________________________________

Epoch 2/25
Training Set | Loss: 1.5593, Top-1 Accuracy: 45.00%, Top-5 Accuracy: 88.33%
Validation Set | Loss: 2.9701, Top-1 Accuracy: 21.37%, Top-5 Accuracy: 54.78%
Best Validation Loss (Up Until Now): 2.4488
_________________________________________________________________________________________________________

Epoch 3/25
Training Set | Loss: 1.2358, Top-1 Accuracy: 63.33%, Top-5 Accuracy: 98.33%
Validation Set | Loss: 3.4870, Top-1 Accuracy: 24.84%, Top-5 Accuracy: 66.54%
Best Validation Loss (Up Until Now): 2.4488
_______________________________________________________________________

Prior to applying augmentation to the extremely low-data training split (i.e., 0.1% of the training set), the model achieves a classification accuracy of $59.07\%$, a macro $\mathsf F_1$ score of $0.5731$, and an optimal validation (cross-entropy) loss of $1.6531$ at epoch 22.